# Explainable Customer Churn Prediction and Retention Strategy
## Model Comparison: Logistic Regression, Random Forest, and XGBoost

## 1. Import Libraries
## 2. Load Data
## 3. Data Cleaning and Preparation
## 4. Exploratory Data Analysis
## 5. Feature Engineering
## 6. Train-Test Split
## 7. Logistic Regression
## 8. Random Forest
## 9. XGBoost
## 10. Model Comparison
## 11. Threshold Optimization
## 12. Explainability
## 13. Retention Strategy
## 14. Export Outputs

In [1]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn xgboost matplotlib shap joblib openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\emeka\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

print("All libraries imported successfully")

C:\Users\emeka\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully


In [3]:
df = pd.read_excel("../data/Telco_churn.xlsx")
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [4]:
df.shape

(7043, 33)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   str    
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   str    
 3   State              7043 non-null   str    
 4   City               7043 non-null   str    
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   str    
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   str    
 10  Senior Citizen     7043 non-null   str    
 11  Partner            7043 non-null   str    
 12  Dependents         7043 non-null   str    
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   str    
 15  Multiple Lines     7043 non-null   str    
 16  Internet Service   7043 non-null   

In [6]:
df.isnull().sum()

CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
dtype: int64

In [7]:
df_clean = df.copy()

# Drop irrelevant columns
cols_to_drop = [
    'CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
    'Lat Long', 'Latitude', 'Longitude'
]

df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')

In [8]:
# fix Total charges i.e from text to numeric
df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce')

In [9]:
df_clean = df_clean.dropna()

In [10]:
# Changing the datatype of target variable to numeric
df_clean['Churn Value'] = df_clean['Churn Value'].astype(int)

In [11]:
# Drop leakage columns
df_clean = df_clean.drop(columns=[
    'Churn Label',
    'Churn Score',
    'Churn Reason'
], errors='ignore')

In [12]:
df_clean.shape

(1869, 21)

In [13]:
df_clean.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,3239
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,2701
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,5372
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,...,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1,5003
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,...,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1,5340


In [14]:
df_clean.to_csv("../outputs/data/df_clean.csv", index=False)

In [15]:
df_clean.shape

(1869, 21)

In [16]:
df_clean = df.copy()

In [17]:
df_clean.shape

(7043, 33)

In [18]:
df_clean['Churn Value'] = df_clean['Churn Value'].astype(int)

In [19]:
df_clean = df_clean.drop(columns=[
    'Churn Label',
    'Churn Score',
    'Churn Reason'
], errors='ignore')

In [20]:
df_clean.shape

(7043, 30)

In [21]:
# Drop irrelevant columns
cols_to_drop = [
    'CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
    'Lat Long', 'Latitude', 'Longitude'
]

df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')

In [22]:
df_clean.shape

(7043, 21)

In [23]:
df_clean.to_csv("../outputs/data/df_clean.csv", index=False)
X = df_clean.drop('Churn Value', axis=1)
y = df_clean['Churn Value']

In [24]:
categorical_cols = X.select_dtypes(include=['object']).columns
categorical_cols

C:\Users\emeka\AppData\Local\Temp\ipykernel_12464\2870865712.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns


Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Total Charges'],
      dtype='str')

In [25]:
# Apply One Hot Encoding
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

In [26]:
X_encoded.shape

(7043, 6560)

In [27]:
df_clean = df.copy()
cols_to_drop = [
    'CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
    'Lat Long', 'Latitude', 'Longitude',
    'Churn Label', 'Churn Score', 'Churn Reason'
]

df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')
df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce')
df_clean = df_clean.dropna()
X = df_clean.drop('Churn Value', axis=1)
y = df_clean['Churn Value']
categorical_cols = X.select_dtypes(include=['object']).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
X_encoded.shape



C:\Users\emeka\AppData\Local\Temp\ipykernel_12464\2175866638.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns


(7032, 31)

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [29]:
# Check Split
X_train.shape, X_test.shape

((5625, 31), (1407, 31))

In [30]:
from sklearn.preprocessing import StandardScaler

numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [31]:
# Save Split data
X_train.to_csv("../outputs/data/X_train.csv", index=False)
X_test.to_csv("../outputs/data/X_test.csv", index=False)
y_train.to_csv("../outputs/data/y_train.csv", index=False)
y_test.to_csv("../outputs/data/y_test.csv", index=False)

In [32]:
X_train.shape, X_test.shape

((5625, 31), (1407, 31))

In [33]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)

lr.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [34]:
# make predixtion
y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lr))

              precision    recall  f1-score   support

           0       0.90      0.71      0.79      1033
           1       0.50      0.79      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.75      1407

ROC-AUC: 0.8425798903562128


In [35]:
lr_auc = roc_auc_score(y_test, y_proba_lr)

## LR Interpretation
Good at detecting churners (high recall)
But has false positives (lower precision)


In [36]:
## Train baseline Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [37]:
## Evaluate RF

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1033
           1       0.64      0.51      0.57       374

    accuracy                           0.79      1407
   macro avg       0.74      0.70      0.72      1407
weighted avg       0.78      0.79      0.79      1407

ROC-AUC: 0.8323324929725475


In [ ]:
## Parameter Tuning RF
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt']
}

rf_model = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)


Fitting 3 folds for each of 24 candidates, totalling 72 fits


In [ ]:
## Best Param tuning RF
best_rf = grid_search.best_estimator_
print(grid_search.best_params_)

In [ ]:
## Evaluate Tuned RF
y_pred_rf_best = best_rf.predict(X_test)
y_proba_rf_best = best_rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf_best))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf_best))

In [ ]:
#'Train Baseline XGBoost
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

In [ ]:
## Prediction
y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

## Evaluate
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))

In [ ]:
## XGBoost Tuning(Define the parameter grid)
from sklearn.model_selection import GridSearchCV

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}

In [ ]:
xgb_model = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

In [ ]:
grid_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train, y_train)

In [ ]:
best_xgb = grid_xgb.best_estimator_
print(grid_xgb.best_params_)

In [ ]:
## Evaluate Tuned Parameter XGBOOST
y_pred_xgb_best = best_xgb.predict(X_test)
y_proba_xgb_best = best_xgb.predict_proba(X_test)[:, 1]

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_xgb_best))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb_best))

## Business Decision
Although XGBoost achieved a marginally higher ROC-AUC, Random Forest was selected as the final model due to its superior balance between recall and precision. 

In a churn prediction context, missing potential churners carries a higher business cost than false positives. Random Forest achieved a significantly higher recall (0.77 vs 0.56), ensuring more at-risk customers are identified for proactive retention strategies, while maintaining a strong F1 score for overall model effectiveness.

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

feature_importance.head()

In [ ]:
feature_importance.to_csv("../outputs/features/feature_importance.csv", index=False)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

threshold_results = []

for t in thresholds:
    y_pred_t = (y_proba_rf_best >= t).astype(int)
    
    precision = precision_score(y_test, y_pred_t)
    recall = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    
    threshold_results.append({
        'Threshold': t,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })

threshold_df = pd.DataFrame(threshold_results)
threshold_df

## Choose a Threshold
A threshold of 0.50 was selected as it provides the optimal balance between precision and recall. This ensures that a significant proportion of at-risk customers are identified for intervention while maintaining manageable false positive rates, supporting efficient allocation of retention resources.

In [ ]:
## Creating a Risk Segmentation(High,medium and Low)
df_strategy = X_test.copy()

df_strategy['Actual'] = y_test
df_strategy['Churn_Probability'] = y_proba_rf_best
def assign_risk(p):
    if p >= 0.6:
        return 'High Risk'
    elif p >= 0.4:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df_strategy['Risk_Level'] = df_strategy['Churn_Probability'].apply(assign_risk)

In [ ]:
## Add Business Actions
def action_map(risk):
    if risk == 'High Risk':
        return 'Immediate retention offer'
    elif risk == 'Medium Risk':
        return 'Engagement campaign'
    else:
        return 'Maintain relationship'

df_strategy['Recommended_Action'] = df_strategy['Risk_Level'].apply(action_map)

In [ ]:
df_strategy[['Churn_Probability','Risk_Level','Recommended_Action']].head()

In [ ]:
df_strategy.to_csv("../outputs/data/df_strategy.csv", index=False)

In [ ]:
import shap

In [ ]:
##Using BEST Model Ramdom Forest)
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

In [ ]:
import numpy as np

print(type(shap_values))
print(np.array(shap_values).shape)
print(X_test.shape)

In [ ]:
shap_values_churn = shap_values[:, :, 1]

In [ ]:
shap.summary_plot(shap_values_churn, X_test)

In [ ]:
## Bar plot version
shap.summary_plot(shap_values_churn, X_test, plot_type="bar")

In [ ]:
import matplotlib.pyplot as plt

shap.summary_plot(shap_values_churn, X_test, show=False)
plt.savefig("../outputs/figures/shap_summary.png", bbox_inches='tight')
plt.close()

In [ ]:
import joblib

joblib.dump(best_rf, "../models/random_forest_final.pkl")

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Precision': [0.50, 0.54, 0.64],
    'Recall': [0.79, 0.77, 0.56],
    'F1 Score': [0.61, 0.64, 0.60],
    'ROC-AUC': [0.8426, 0.8446, 0.8486]
})

In [ ]:
comparison.to_csv("../outputs/metrics/model_comparison.csv", index=False)

In [ ]:
threshold_df.to_csv("../outputs/metrics/threshold_optimization.csv", index=False)

In [ ]:
import numpy as np

shap_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean_SHAP_Impact': np.abs(shap_values_churn).mean(axis=0)
}).sort_values(by='Mean_SHAP_Impact', ascending=False)

shap_importance.head()

In [ ]:
shap_importance.to_csv("../outputs/features/shap_feature_importance.csv", index=False)

In [ ]:
df_full = X_encoded.copy()

df_full['Actual'] = y
df_full['Churn_Probability'] = best_rf.predict_proba(X_encoded)[:, 1]

In [ ]:
df_full.to_csv("../outputs/data/powerbi_dataset.csv", index=False)

In [ ]:
df_full = df_clean.copy()
df_full['Churn_Probability'] = best_rf.predict_proba(X_encoded)[:, 1]
def assign_risk(p):
    if p >= 0.6:
        return 'High Risk'
    elif p >= 0.4:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df_full['Risk_Level'] = df_full['Churn_Probability'].apply(assign_risk)
def action_map(risk):
    if risk == 'High Risk':
        return 'Immediate retention offer'
    elif risk == 'Medium Risk':
        return 'Engagement campaign'
    else:
        return 'Maintain relationship'

df_full['Recommended_Action'] = df_full['Risk_Level'].apply(action_map)
df_full.to_csv("../outputs/powerbi_dataset.csv", index=False)